In [1]:
# Install the required packages for Vespa Python client and Docker
# !pip install pyvespa docker

In [68]:
import random
import pickle
from vespa.deployment import VespaCloud
import json
import unicodedata
from dataclasses import dataclass
from typing import Callable, Optional, Iterable, Dict
from vespa.application import Vespa
from time import time
from tqdm.auto import tqdm
import nest_asyncio
from vespa.evaluation import VespaEvaluator
from vespa.package import (
    ApplicationPackage,
    Field,
    Schema,
    Document,
    HNSW,
    RankProfile,
    Component,
    Parameter,
    FieldSet,
    GlobalPhaseRanking,
    Function,
    OnnxModel,
)

In [64]:
import requests
from pathlib import Path

url = "https://huggingface.co/mixedbread-ai/mxbai-rerank-xsmall-v1/resolve/main/onnx/model_quantized.onnx"
local_model_path = "model/model_quantized.onnx"

r = requests.get(url)
# Create path if it doesn't exist
Path(local_model_path).parent.mkdir(parents=True, exist_ok=True)
with open(local_model_path, "wb") as f:
    f.write(r.content)
    print(f"Downloaded model to {local_model_path}")

Downloaded model to model/model_quantized.onnx


In [65]:
# nome da aplicação
application = "myapp"

In [66]:
# Aqui podemos alterar o esquema caso necessário, mas por enquanto vamos usar o esquema padrão do tutorial.
package = ApplicationPackage(
    name=application,
    schema=[
        Schema(
            name="doc",
            document=Document(
                fields=[
                    Field(name="id", type="string", indexing=["summary"]),
                    Field(
                        name="body",
                        type="string",
                        indexing=["index", "summary"],
                        index="enable-bm25",
                        bolding=True,
                    ),
                    Field(
                        name="embedding",
                        type="tensor(x[384])",
                        indexing=[
                            'input body',
                            "embed",
                            "index",
                            "attribute",
                        ],
                        ann=HNSW(distance_metric="angular"),
                        is_document_field=False,
                    ),
                ]
            ),
            fieldsets=[FieldSet(name="default", fields=["body"])],
            rank_profiles=[
                RankProfile(
                    name="bm25",
                    inputs=[("query(q)", "tensor(x[384])")],
                    functions=[
                        Function(name="bm25sum", expression="bm25(body)")
                    ],
                    first_phase="bm25sum",
                ),
                RankProfile(
                    name="semantic",
                    inputs=[("query(q)", "tensor(x[384])")],
                    first_phase="closeness(field, embedding)",
                ),
                RankProfile(
                    name="fusion",
                    inherits="bm25",
                    inputs=[("query(q)", "tensor(x[384])")],
                    first_phase="closeness(field, embedding)",
                    global_phase=GlobalPhaseRanking(
                        expression="reciprocal_rank_fusion(bm25sum, closeness(field, embedding))",
                        rerank_count=1000,
                    ),
                ),
            ],
        )
    ],
    components=[
        Component(
            id="e5",
            type="hugging-face-embedder",
            parameters=[
                Parameter(
                    "transformer-model",
                    {
                        "url": "https://github.com/vespa-engine/sample-apps/raw/master/examples/model-exporting/model/e5-small-v2-int8.onnx"
                    },
                ),
                Parameter(
                    "tokenizer-model",
                    {
                        "url": "https://raw.githubusercontent.com/vespa-engine/sample-apps/master/examples/model-exporting/model/tokenizer.json"
                    },
                ),
            ],
        )
    ],
)

In [89]:

schema = Schema(
    name="doc",
    mode="index",
    document=Document(
        fields=[
            Field(name="body", type="string", indexing=["index", "summary"], index="enable-bm25"),
            Field(
                name="text_tokens",
                type="tensor<float>(d0[512])",
                indexing=["input body", "embed tokenizer", "attribute", "summary"],
                is_document_field=False,
            ),
            Field(name="id", type="string", indexing=["summary", "attribute"]),
        ],
    ),
    fieldsets=[FieldSet(name="default", fields=["body"])],
    models=[
        OnnxModel(
            model_name="crossencoder",
            model_file_path=f"{local_model_path}",
            inputs={
                "input_ids": "input_ids",
                "attention_mask": "attention_mask",
            },
            outputs={"logits": "logits"},
        )
    ],
    rank_profiles=[
        RankProfile(name="bm25", first_phase="bm25(body)"),
        RankProfile(
            name="reranking",
            inherits="default",
            inputs=[("query(q)", "tensor<float>(d0[64])")],
            functions=[
                Function(
                    name="input_ids",
                    expression="customTokenInputIds(1, 2, 512, query(q), attribute(text_tokens))",
                ),
                Function(
                    name="attention_mask",
                    expression="tokenAttentionMask(512, query(q), attribute(text_tokens))",
                ),
            ],
            first_phase="bm25(body)",
            global_phase=GlobalPhaseRanking(
                rerank_count=100,
                expression="sigmoid(onnx(crossencoder).logits{d0:0,d1:0})",
            ),
            summary_features=[
                "query(q)",
                "input_ids",
                "attention_mask",
                "onnx(crossencoder).logits",
            ],
        ),
    ],
)


In [84]:
app_package = ApplicationPackage(
    name=application,
    schema=[schema],
    components=[
        Component(
            # See https://docs.vespa.ai/en/reference/embedding-reference.html#huggingface-tokenizer-embedder
            id="tokenizer",
            type="hugging-face-tokenizer",
            parameters=[
                Parameter(
                    "model",
                    {
                        "url": "https://huggingface.co/mixedbread-ai/mxbai-rerank-xsmall-v1/raw/main/tokenizer.json"
                    },
                ),
            ],
        )
    ],
)

In [90]:

import os
import docker

# Set the DOCKER_HOST environment variable to point to the Docker Desktop socket
# This is necessary because Docker Desktop on Linux uses a different socket path
# from the default, and pyvespa/docker-py may not pick it up automatically.
docker_desktop_socket_path = os.path.expanduser("~/.docker/desktop/docker.sock")
if os.path.exists(docker_desktop_socket_path):
    os.environ['DOCKER_HOST'] = f"unix://{docker_desktop_socket_path}"

# Verify that we can connect to the Docker daemon
try:
    client = docker.from_env()
    print("Successfully connected to Docker daemon.")
    print(f"Docker version: {client.version()['Version']}")
except Exception as e:
    print(f"Failed to connect to Docker daemon: {e}")
    print("\nPlease ensure Docker Desktop is running.")



Successfully connected to Docker daemon.
Docker version: 28.1.1


In [91]:
# Sempre que essa célula der erro execute o comando vespa auth login no terminal
# para autenticar novamente com o Vespa Cloud
# vespa_cloud = VespaCloud(
#     tenant=tenant_name,
#     application=application,
#     application_package=package,
# )

from vespa.deployment import VespaDocker

vespa_docker = VespaDocker()
app = vespa_docker.deploy(application_package=app_package)

Waiting for configuration server, 0/60 seconds...
Waiting for configuration server, 5/60 seconds...
Waiting for configuration server, 5/60 seconds...
Waiting for application to come up, 0/300 seconds.
Waiting for application to come up, 0/300 seconds.
Application is up!
Finished deployment.
Application is up!
Finished deployment.


In [75]:
# Splita as queries de treinamento e teste

random.seed(42)

def load_dataset(input_file):
    with open(input_file, 'rb') as f:
        return pickle.load(f)

data_set = "../subset_msmarco_train_0.01_99.pkl"

data = load_dataset(data_set)
queries = data["queries"]
documents = data["docs"]
qrels = data["qrels"]

# Split the queries (queries is a dictionary of {query_id: query_object})
query_ids = list(queries.keys())  # List of query IDs

# Shuffle query IDs to ensure a random split
random.shuffle(query_ids)

# Split into 80% for training, 20% for validation
split_ratio = 0.8
train_query_ids = query_ids[:int(len(query_ids) * split_ratio)]
test_query_ids = query_ids[int(len(query_ids) * split_ratio):]

train_queries = {qid: queries[qid] for qid in train_query_ids}
test_queries = {qid: queries[qid] for qid in test_query_ids}

In [76]:
# remove control characters from the text to avoid issues with Vespa Feed

def remove_control_characters(text: str) -> str:
    """Remove caracteres de controle e não imprimíveis do texto."""
    return ''.join(
        ch for ch in text
        if unicodedata.category(ch)[0] != 'C' or ch in '\n\t\r'
    )

In [ ]:
# Saves the data into a json format that Vespa can understand possivelmente we can remove this saving step 
# and create the json on memory just before the feed step

# namespace = "default"
# doctype = "doc"

# vespa_docs = []

# for doc_id, doc_obj in documents.items():
#     vespa_doc = {
#         "put": f"id:{namespace}:{doctype}::{doc_id}",
#         "fields": {
#             "id": str(doc_id),
#             "body": remove_control_characters(doc_obj.text),
#         }
#     }
#     vespa_docs.append(vespa_doc)

# feed_file = "vespa_feed.json"

# with open(feed_file, "w", encoding="utf-8") as f:
#     json.dump(vespa_docs, f, ensure_ascii=False)

# print(f"✅ {len(vespa_docs)} documentos salvos em {feed_file}")

✅ 277168 documentos salvos em vespa_feed.json


In [77]:
# Define feed parameters for the Vespa application
@dataclass
class FeedParams:
    name: str
    num_docs: int
    max_connections: int
    function_name: str
    max_workers: Optional[int] = None
    max_queue_size: Optional[int] = None


@dataclass
class FeedResult(FeedParams):
    feed_time: Optional[float] = None

In [ ]:
# This is necessary to avoid issues with asyncio and Jupyter notebooks
nest_asyncio.apply()

In [ ]:
# # Asynchronous feed function that sends documents to Vespa
# params = FeedParams(
#     name="full_async_feed",
#     function_name="feed_async_iterable",
#     num_docs=0,  # não é usado aqui
#     max_connections=16,
#     max_workers=32,
#     max_queue_size=5000,
# )

# # Load the documents from the JSON file
# with open("vespa_feed.json", "r", encoding="utf-8") as f:
#     data_list = json.load(f)

# # Prepare the dataset for feeding into Vespa
# dataset = [
#     {"id": item["fields"]["id"], "fields": item["fields"]}
#     for item in tqdm(data_list, desc="🔄 Preparando documentos para envio")
# ]

# # Feed the dataset into Vespa asynchronously
# with tqdm(total=len(dataset), desc="📤 Enviando documentos para Vespa") as pbar:
#     def progress_callback(response, doc_id):
#         pbar.update(1)
#         if not response.is_successful():
#             print(f"❌ Erro ao enviar {doc_id}: {response.status_code} - {response.get_json()}")

#     start = time()
#     app.feed_async_iterable(
#         dataset,
#         schema="doc",
#         namespace="pyvespa-feed",
#         operation_type="feed",
#         max_queue_size=params.max_queue_size,
#         max_workers=params.max_workers,
#         max_connections=params.max_connections,
#         callback=progress_callback,
#     )
#     duration = time() - start

# print(f"✅ Feed finalizado em {duration:.2f} segundos") 


In [ ]:
test_queries_dict = {
    q.query_id: q.text
    for q in test_queries.values()
}

relevant_docs = dict()
for qrel in qrels:
    relevant_docs[qrel.query_id] = relevant_docs.get(qrel.query_id, set())
    relevant_docs[qrel.query_id].add(qrel.doc_id)

search_methods = [
    ("Plain Keyword", "select * from sources * where userQuery()"),
    ("Plain Semantic", "select* from sources * where ({targetHits:1000}nearestNeighbor(embedding,q))"),
    ("Hybrid OR", "select * from sources * where userQuery() or ({targetHits:1000}nearestNeighbor(embedding,q))")
]

ranking_profiles = ["bm25", "semantic", "fusion"]

def create_vespa_query_fn(yql_template: str, ranking_profile: str) -> callable:
    """
    Vespa query function generator.
    Generates a query function that can be used with VespaEvaluator.
    """
    def query_fn(query_text: str, top_k: int) -> dict:
        query_body = {
            "yql": f"{yql_template} limit {top_k}",
            "query": query_text,
            "ranking": ranking_profile,
        }
        if ranking_profile in ["semantic", "fusion"]:
            query_body["ranking.features.query(q)"] = f"embed({query_text})"
        return query_body
    return query_fn

for search_name, yql in search_methods:
    for ranking in ranking_profiles:
        print(f"--- '{search_name}' '{ranking}' ---")

        vespa_query_fn = create_vespa_query_fn(yql_template=yql, ranking_profile=ranking)

        evaluator = VespaEvaluator(
            queries=test_queries_dict,
            relevant_docs=relevant_docs,
            vespa_query_fn=vespa_query_fn,
            app=app,
            name=f"test-run-{search_name.lower().replace(' ', '_')}-{ranking}",
            accuracy_at_k=[10],
            precision_recall_at_k=[100],
            mrr_at_k=[10],
            ndcg_at_k=[10],
            write_csv=True
        )

        results = evaluator.run()

        print(f"Results for '{search_name}' and '{ranking}':")
        print("Primary Metric:", evaluator.primary_metric)
        print("Results:", results)
        print("-" * 50 + "\n")

Results for bm25:
Primary metric: ndcg@10
All results: {'accuracy@10': 0.7279279279279279, 'precision@100': 0.009045045045044938, 'recall@100': 0.8675675675675676, 'mrr@10': 0.5304332904332902, 'ndcg@10': 0.5714938525410317, 'map@100': 0.5305388049322048, 'searchtime_avg': 0.027828828828828752, 'searchtime_q50': 0.025, 'searchtime_q90': 0.049600000000000026, 'searchtime_q95': 0.056}
Results for semantic:
Primary metric: ndcg@10
All results: {'accuracy@10': 0.5891891891891892, 'precision@100': 0.006108108108108061, 'recall@100': 0.5864864864864865, 'mrr@10': 0.47457886457886433, 'ndcg@10': 0.4999961431385134, 'map@100': 0.47056199056199033, 'searchtime_avg': 0.014461261261261226, 'searchtime_q50': 0.012, 'searchtime_q90': 0.027600000000000024, 'searchtime_q95': 0.033299999999999955}
Results for fusion:
Primary metric: ndcg@10
All results: {'accuracy@10': 0.863063063063063, 'precision@100': 0.009783783783783662, 'recall@100': 0.9378378378378378, 'mrr@10': 0.6838109538109535, 'ndcg@10': 0

In [94]:
test_queries_dict

{'238659': 'how long after a bankruptcy can you refinance',
 '59728': 'calories in a peppermint lifesaver',
 '452869': 'message broker service definition',
 '992173': 'where is san augustine texas',
 '66176': 'can cycling help reduce belly fat',
 '995442': 'where is the environment variable file?',
 '453707': 'mifid what does multilateral mean',
 '127359': 'define splendor',
 '131800': 'definition of a process agent',
 '841729': 'what is the province that borders yukon and alberta',
 '191319': 'four types of sorrow an what is needed for absolution',
 '403094': 'is apple parts produced by samsung',
 '339093': 'how soon after conception symptoms',
 '738584': 'what is dental ppo insurance',
 '618544': 'what dictates formation of random coil',
 '706595': 'what is a xsd to xml',
 '440767': 'list of food that are high in acid',
 '678679': 'what is a computer trackball',
 '747646': 'what is fl state sales tax',
 '856089': 'what is time difference nyc and london',
 '506144': 'symbolism of crow

In [95]:
test_queries_dict = {
    q.query_id: q.text
    for q in test_queries.values()
}

relevant_docs = dict()
for qrel in qrels:
    relevant_docs[qrel.query_id] = relevant_docs.get(qrel.query_id, set())
    relevant_docs[qrel.query_id].add(qrel.doc_id)

# Vespa query function for cross-encoder reranking
def create_crossencoder_query_fn(top_k=10):
    def query_fn(query_text: str, k: int = top_k) -> dict:
        return {
            "yql": f"select * from sources * where userQuery();",
            "query": query_text,
            "ranking": "bm25",
            "ranking.listFeatures": "true",
            "presentation.timing": "true",
        }
    return query_fn

vespa_query_fn = create_crossencoder_query_fn(top_k=10)

# Evaluate using VespaEvaluator with the reranking profile
evaluator = VespaEvaluator(
    queries=test_queries_dict,
    relevant_docs=relevant_docs,
    vespa_query_fn=vespa_query_fn,
    app=app,
    name="test-run-crossencoder-reranking",
    accuracy_at_k=[10],
    precision_recall_at_k=[10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    write_csv=True
)

results = evaluator.run()

print("Results for cross-encoder reranking:")
print("Primary Metric:", evaluator.primary_metric)
print("Results:", results)

Results for cross-encoder reranking:
Primary Metric: ndcg@10
Results: {'accuracy@10': 0.0, 'precision@10': 0.0, 'recall@10': 0.0, 'mrr@10': 0.0, 'ndcg@10': 0.0, 'map@100': 0.0, 'searchtime_avg': 0.0026522522522522524, 'searchtime_q50': 0.002, 'searchtime_q90': 0.005, 'searchtime_q95': 0.006}
